# Evaluación automática

Funciona con MLflow local o Unity Catalog según `IRIS_RUNTIME`.

In [ ]:
import json
from dataclasses import replace
import os
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException
from sklearn.model_selection import train_test_split

from iris_mlflow_utils import (
    build_evaluation_artifacts,
    build_probability_metrics,
    build_runtime_config,
    build_deployment_config,
    detect_runtime,
    ensure_mlflow_experiment,
    evaluate_model,
    evaluate_promotion_gate,
    get_model_evaluation_metrics,
    load_dataset_for_runtime,
)

runtime_mode = detect_runtime()
config = build_runtime_config(model_slug='random_forest')
deployment_config = build_deployment_config()
if runtime_mode == 'databricks':
    dbutils.widgets.text('model_name', deployment_config.model_name)
    dbutils.widgets.text('model_version', '')
    model_name = dbutils.widgets.get('model_name')
    model_version = dbutils.widgets.get('model_version')
else:
    model_name = deployment_config.model_name
    model_version = os.getenv('IRIS_MODEL_VERSION', '')
if config.tracking_uri:
    mlflow.set_tracking_uri(config.tracking_uri)
mlflow.set_registry_uri(config.registry_uri)
ensure_mlflow_experiment(config.experiment_name, tracking_uri=config.tracking_uri, artifact_location=config.artifact_location)
client = MlflowClient(tracking_uri=config.tracking_uri, registry_uri=config.registry_uri)
if not model_version:
    versions = list(client.search_model_versions(f"name='{model_name}'"))
    if not versions:
        raise RuntimeError(f'No hay versiones para {model_name}.')
    model_version = str(max(versions, key=lambda item: int(item.version)).version)
version = client.get_model_version(model_name, model_version)
logged_model_id = getattr(version, 'model_id', None)
if not logged_model_id and str(version.source).startswith('models:/'):
    logged_model_id = str(version.source).removeprefix('models:/').strip('/')
model_type = version.tags.get('model_type', 'random_forest')
feature_table_version = version.tags.get('feature_table_version', '')
if runtime_mode == 'databricks' and not feature_table_version:
    raise RuntimeError('La versión candidata no tiene feature_table_version reproducible.')
config = replace(config, feature_table_version=feature_table_version or 'local')
model_uri = f'models:/{model_name}/{model_version}'
if model_type == 'xgboost':
    model = mlflow.xgboost.load_model(model_uri)
else:
    model = mlflow.sklearn.load_model(model_uri)


In [ ]:
dataset = load_dataset_for_runtime(
    runtime_mode=runtime_mode,
    spark=globals().get('spark'),
    config=config,
    table_version=None if runtime_mode == 'local' else feature_table_version,
)
x_train, x_test, y_train, y_test = train_test_split(
    dataset.features, dataset.target, test_size=config.test_size,
    random_state=config.random_state, stratify=dataset.target,
)
x_test = x_test.astype('float64')
result = evaluate_model(model, x_test, y_test, list(range(len(dataset.classes))))
result.metrics.update(build_probability_metrics(result, y_test, list(range(len(dataset.classes)))))
metrics = {f'test_{key}': value for key, value in result.metrics.items()}
with mlflow.start_run(run_name=f'evaluate-{model_name.split(".")[-1]}-{model_version}') as evaluation_run:
    evaluation_run_id = evaluation_run.info.run_id
    mlflow.set_tags({'runtime': runtime_mode, 'model_name': model_name, 'model_version': model_version, 'model_id': logged_model_id or '', 'model_type': model_type, 'feature_table': config.feature_table, 'feature_table_version': feature_table_version or 'local', 'stage': 'evaluation'})
    with tempfile.TemporaryDirectory() as directory:
        output_dir = Path(directory) / 'evaluation'
        paths = build_evaluation_artifacts(
            model, result, labels=list(range(len(dataset.classes))),
            class_names=dataset.classes, features=x_test, target=y_test, output_dir=output_dir,
        )
        predictions = x_test.copy()
        predictions['actual'] = y_test
        predictions['prediction'] = result.predictions
        if result.probabilities is not None:
            for index, class_name in enumerate(dataset.classes):
                predictions[f'probability_{class_name}'] = result.probabilities[:, index]
        predictions.to_parquet(output_dir / 'predictions.parquet', index=False)
        (output_dir / 'metrics_summary.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
        (output_dir / 'classification_report.json').write_text(json.dumps(result.report, indent=2, default=float), encoding='utf-8')
        classification_by_class = {class_name: result.report.get(str(index), {}) for index, class_name in enumerate(dataset.classes)}
        (output_dir / 'classification_by_class.json').write_text(json.dumps(classification_by_class, indent=2, default=float), encoding='utf-8')
        (output_dir / 'class_mapping.json').write_text(json.dumps({str(i): name for i, name in enumerate(dataset.classes)}, indent=2), encoding='utf-8')
        (output_dir / 'input_schema.json').write_text(json.dumps({str(column): str(dtype) for column, dtype in x_test.dtypes.items()}, indent=2), encoding='utf-8')
        for path in paths.values():
            mlflow.log_artifact(str(path), artifact_path='evaluation')
        for name in ('predictions.parquet', 'metrics_summary.json', 'classification_report.json', 'classification_by_class.json', 'class_mapping.json', 'input_schema.json'):
            mlflow.log_artifact(str(output_dir / name), artifact_path='evaluation')
        mlflow.log_metrics(metrics, model_id=logged_model_id)
champion_metrics = None
comparison_run_id = evaluation_run_id
try:
    champion = client.get_model_version_by_alias(model_name, deployment_config.champion_alias)
    champion_feature_version = champion.tags.get('feature_table_version', '')
    if champion_feature_version == (feature_table_version or 'local'):
        champion_metrics, comparison_run_id = get_model_evaluation_metrics(client, model_name=model_name, model_version=champion.version)
    else:
        champion_type = champion.tags.get('model_type', 'random_forest')
        champion_uri = f'models:/{model_name}/{champion.version}'
        champion_model = mlflow.xgboost.load_model(champion_uri) if champion_type == 'xgboost' else mlflow.sklearn.load_model(champion_uri)
        champion_result = evaluate_model(champion_model, x_test, y_test, list(range(len(dataset.classes))))
        champion_result.metrics.update(build_probability_metrics(champion_result, y_test, list(range(len(dataset.classes)))))
        champion_metrics = {f'test_{key}': value for key, value in champion_result.metrics.items()}
        with mlflow.start_run(run_name=f'compare-champion-{champion.version}-candidate-{model_version}') as comparison_run:
            comparison_run_id = comparison_run.info.run_id
            mlflow.set_tags({'stage': 'champion_comparison', 'model_name': model_name, 'champion_version': str(champion.version), 'candidate_version': model_version, 'feature_table_version': feature_table_version})
            mlflow.log_metrics({f'champion_{key}': value for key, value in champion_metrics.items()})
except MlflowException as error:
    missing_alias = any(marker in str(error).lower() for marker in ('does not exist', 'not found', 'resource_does_not_exist'))
    if not missing_alias:
        raise
    champion_metrics = None
decision = evaluate_promotion_gate(metrics, champion_metrics, deployment_config)
client.set_model_version_tag(model_name, model_version, 'evaluation_status', 'passed' if decision.passed else 'failed')
client.set_model_version_tag(model_name, model_version, 'gate_status', 'passed' if decision.passed else 'failed')
client.set_model_version_tag(model_name, model_version, 'feature_table_version', feature_table_version or 'local')
client.set_model_version_tag(model_name, model_version, 'evaluation_decision', json.dumps(decision.as_dict()))
client.set_model_version_tag(model_name, model_version, 'evaluation_run_id', evaluation_run_id)
client.set_model_version_tag(model_name, model_version, 'comparison_run_id', comparison_run_id)
if logged_model_id:
    client.set_model_version_tag(model_name, model_version, 'evaluation_model_id', logged_model_id)
print({'runtime': runtime_mode, 'model_name': model_name, 'model_version': model_version, **decision.as_dict()})
if not decision.passed:
    raise RuntimeError(f'La versión no supera los gates: {decision.reason}')
